#Logs-

1. secret info manual vec db created
2. rag response format neo4j graph created

#Installations

In [ ]:
!pip install langchain_community
!pip install langchain_google_genai
!pip install langchain_openai
!pip install langchain_groq
!pip install faiss--cpu
!pip install langchain_experimental
!pip install langchain_neo4j
!pip install pypdf
!pip install neo4j

#Environment variables and API Keys

In [ ]:
import os
os.environ['GOOGLE_API_KEY']='' #api_key_1
os.environ['OPENAI_API_KEY']='' #api_key_2
os.environ['GROQ_API_KEY']="" #api_key_3

#Preparing the Secret info manual- This will be a vector database

embedding model- "models/embedding-001"

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
secret_info_manual=PyPDFLoader("/content/SECRET INFO MANUAL.pdf")
secret_info_manual=secret_info_manual.load()

from langchain.text_splitter import RecursiveCharacterTextSplitter
rcts_1=RecursiveCharacterTextSplitter(chunk_size=700,chunk_overlap=100)
secret_info_manual=rcts_1.split_documents(secret_info_manual)

from langchain_community.vectorstores import FAISS
secret_manual_info_db=FAISS.from_documents(secret_info_manual,embeddings)

In [ ]:
secret_manual_info_db.similarity_search("What is the Layered Cipher Code system used by RAW?")

[Document(id='9af806d4-2ac6-4d7c-9727-bd21bb5c2537', metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-04-13T07:20:36+00:00', 'author': 'Aman Agarwal', 'moddate': '2025-04-13T07:20:36+00:00', 'source': '/content/SECRET INFO MANUAL.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='every message, every identity must be verified, cross-validated, and encrypted \nusing multi-layered authentication techniques. The methodologies outlined in this \nmanual are designed to ensure operational integrity, information security, and agent \nsurvival. \nTo confirm authenticity, enter the following cipher into your secured terminal before \nproceeding: \nEK7-ΣΔ19-βXQ//4437 \n \nIf access is denied, cease all attempts immediately and burn this document. \n \nCommunication & Verification Protocols \nLayered Cipher Code (LCC) System \nEvery message exchanged within the RAW network must be encoded using LCC, a \nthree-layer cryptographic 

#Preparing the RAG Case Reponse Framework- This will be a Neo4j Knowledge Graph

Loading RAG Case Response Document

In [ ]:
import os
os.environ['NEO4J_URI']='neo4j+s://03868fe8.databases.neo4j.io'
os.environ['NEO4J_USERNAME']='neo4j'
os.environ['NEO4J_PASSWORD']='3f3nuLgfHVnhBAHWu9etrEJtsXTQB7ByxcaWv58GJyk'

In [ ]:
rag_case_response=PyPDFLoader("/content/RAG CASE RESPONSE FRAMEWORK.pdf")
rag_case_response=rag_case_response.load()

rcts_2=RecursiveCharacterTextSplitter(chunk_size=350,chunk_overlap=50)
rag_case_response=rcts_2.split_documents(rag_case_response)

In [ ]:
rag_case_response

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-04-13T08:24:34+00:00', 'author': 'Aman Agarwal', 'moddate': '2025-04-13T08:24:34+00:00', 'source': '/content/RAG CASE RESPONSE FRAMEWORK.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='RAW Agents’ Query Response Framework (Level 7 \nClassified) \nIssued By: Directorate of Covert Operations \nSecurity Clearance Required: Level 7 and Above \nLast Updated: January 2025 \n \nResponse Protocol Based on Agent Level & Query Type  \nThis document dictates how the RAW Intelligence Retrieval System'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-04-13T08:24:34+00:00', 'author': 'Aman Agarwal', 'moddate': '2025-04-13T08:24:34+00:00', 'source': '/content/RAG CASE RESPONSE FRAMEWORK.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='(RIRS) processes query and generates responses. It estab

LLM to convert the document to a graph document

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_openai = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)

graph_llm = LLMGraphTransformer(
    llm=llm_openai,
    allowed_nodes=[
        "AgentLevel",
        "ResponseRule",
        "Greeting",
        "QueryType",
        "Protocol"
        ],
    allowed_relationships=[
        ("AgentLevel", "GREETS_WITH", "Greeting"),
        ("AgentLevel", "FOLLOWS_RULE", "ResponseRule"),
        ("ResponseRule", "APPLIES_TO", "QueryType"),
        ("ResponseRule", "RETURNS", "Protocol")
    ],
    node_properties=[
        "level",
        "name",
        "codename",
        "text",
        "ruleNumber",
        "description"
        ],
    strict_mode=True,
    additional_instructions="""
- Agent Levels: Identify agent levels (e.g., 'Level 1 - Novice Operative'). Create 'AgentLevel' nodes with 'level' (numeric part only, e.g., 1), 'name' (e.g., 'Novice Operative'), and 'codename' (e.g., 'Shadow Footprint') properties.
- Greetings: Identify greeting text associated with an Agent Level, typically enclosed in double quotes (e.g., '“Salute, Shadow Cadet.”'). Create a 'Greeting' node with the 'text' property containing the greeting *without* the surrounding quotes. Create a 'GREETS_WITH' relationship from the corresponding 'AgentLevel' node to this 'Greeting' node.
- Response Approaches: Identify the response approach text described for an Agent Level (e.g., 'Basic and instructional, like a mentor guiding a trainee.'). Create a 'ResponseApproach' node with the 'text' property containing this description. Create a 'HAS_RESPONSE_APPROACH' relationship from the corresponding 'AgentLevel' node to this 'ResponseApproach' node.
- Response Rules: Identify rules mentioned (e.g., 'Rule 15: If a Level-1 agent asks...'). Create 'ResponseRule' nodes with 'ruleNumber' (numeric part only, e.g., 15) and 'description' properties (the text explaining the rule).
- Rule Conditions (Agent Level):** If a rule description specifies it applies to a certain agent level (e.g., "If a Level-1 agent asks..."), create an 'APPLIES_TO_LEVEL' relationship from the 'ResponseRule' node to the corresponding 'AgentLevel' node.
- Rule Conditions (Query Type/Keywords):** If a rule description specifies it applies to a certain query type or keywords (e.g., '...asks about disguise strategies', '...query contains “Omega Echo”'), create a 'QueryType' node with the 'name' property representing that condition (e.g., 'disguise strategies', 'Omega Echo'). Create an 'APPLIES_TO_QUERYTYPE' relationship from the 'ResponseRule' node to this 'QueryType' node.
- Rule Protocols/Responses: If a rule specifies an exact text response (e.g., 'return: "The shadow moves..."'), create a 'Protocol' node with the 'text' property containing the response *without* surrounding quotes. Create a 'RETURNS' relationship from the 'ResponseRule' node to this 'Protocol' node.
- Create relationships based *only* on the structure and associations presented within the text chunk. Do not infer relationships not explicitly mentioned.
"""
)

graph_rag_case_response = graph_llm.convert_to_graph_documents(rag_case_response)

In [ ]:
graph_rag_case_response

[GraphDocument(nodes=[Node(id='Level 7 - Classified', type='Agentlevel', properties={'level': '7', 'name': 'Classified', 'codename': ''}), Node(id='January 2025', type='Responserule', properties={'rulenumber': '1', 'description': 'This document dictates how the RAW Intelligence Retrieval System'})], relationships=[Relationship(source=Node(id='Level 7 - Classified', type='Agentlevel', properties={}), target=Node(id='January 2025', type='Responserule', properties={}), type='FOLLOWS_RULE', properties={})], source=Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-04-13T08:24:34+00:00', 'author': 'Aman Agarwal', 'moddate': '2025-04-13T08:24:34+00:00', 'source': '/content/RAG CASE RESPONSE FRAMEWORK.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='RAW Agents’ Query Response Framework (Level 7 \nClassified) \nIssued By: Directorate of Covert Operations \nSecurity Clearance Required: Level 7 and Above \nLast Updated

Creating the knowledge graph using the above graph document

In [ ]:
from langchain_community.graphs import Neo4jGraph
knowledge_graph=Neo4jGraph()

knowledge_graph.add_graph_documents(
    graph_documents=graph_rag_case_response,
    baseEntityLabel=True,
    include_source=True
)

Creating a Graph vector DB

In [ ]:
graph_rag_case_response[0].source

Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-04-13T08:24:34+00:00', 'author': 'Aman Agarwal', 'moddate': '2025-04-13T08:24:34+00:00', 'source': '/content/RAG CASE RESPONSE FRAMEWORK.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'id': '9621f8abb39c31e722b310ced738fc5b'}, page_content='RAW Agents’ Query Response Framework (Level 7 \nClassified) \nIssued By: Directorate of Covert Operations \nSecurity Clearance Required: Level 7 and Above \nLast Updated: January 2025 \n \nResponse Protocol Based on Agent Level & Query Type  \nThis document dictates how the RAW Intelligence Retrieval System')

In [ ]:
from langchain_community.vectorstores.neo4j_vector import Neo4jVector
graph_rag_case_response = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    node_label="Document",
    text_node_properties=['text'],
    embedding_node_property="embedding",
    search_type="hybrid"
)

In [ ]:
graph_rag_case_response.similarity_search("Explain disguise strategies for new agents?")

[Document(metadata={'author': 'Aman Agarwal', 'moddate': '2025-04-13T08:24:34+00:00', 'source': '/content/RAG CASE RESPONSE FRAMEWORK.pdf', 'page': 0, 'total_pages': 11, 'producer': 'www.ilovepdf.com', 'creationdate': '2025-04-13T08:24:34+00:00', 'page_label': '1', 'creator': 'Microsoft® Word 2016'}, page_content='\ntext: Agent Classification & Response Style Guide \nAgent Levels & Greeting Protocols \nAgent Level Greeting Style Response Approach \nLevel 1 - Novice \nOperative (Shadow \nFootprint) \n“Salute, Shadow \nCadet.” \nBasic and instructional, like a \nmentor guiding a trainee. \nLevel 2 - Tactical'),
 Document(metadata={'author': 'Aman Agarwal', 'moddate': '2025-04-13T08:24:34+00:00', 'source': '/content/RAG CASE RESPONSE FRAMEWORK.pdf', 'page': 2, 'total_pages': 11, 'producer': 'www.ilovepdf.com', 'creationdate': '2025-04-13T08:24:34+00:00', 'page_label': '3', 'creator': 'Microsoft® Word 2016'}, page_content='\ntext: provide frequency scrambling techniques but include an inbu

#Agentic RAG

To implement Agentic RAG we need the following:

1. Tools
2. Prompt
3. LLM

Tools

We need to convert the Faiss VectorDB and the Neo4j Vector DB to retrievers and these retirevers to tools

In [ ]:
secret_info_manual_retriever=secret_manual_info_db.as_retriever()
graph_rag_case_response_retriever=graph_rag_case_response.as_retriever()

from langchain.tools.retriever import create_retriever_tool
secret_info_manual_tool = create_retriever_tool(
    retriever=secret_info_manual_retriever,
    name="search_raw_agent_secret_manual",
    description="Use this tool to find specific classified operational details, facts, and procedures from the RAW Agents' Secret Information Manual. This includes information on protocols (like LCC, S-29, Project Eclipse, Protocol Zeta-5), communication methods, verification steps (Handshake Protocol), safehouse locations and entry procedures (K-41, H-77, X-17), passcodes, counter-surveillance techniques (Ghost-Step Algorithm), termination protocols, field tactics, extraction methods (Shadow Step), and emergency directives. Ideal for queries asking 'What is..?', 'How does... work?', 'Where is...?', 'What are the steps for...?' regarding specific RAW operations and assets described in the secret manual."
)
graph_rag_case_response_tool = create_retriever_tool(
    retriever=graph_rag_case_response_retriever,
    name="search_agent_response_rules_framework",
    description="Use this tool to understand the rules, guidelines, and protocols for how the RAW Intelligence Retrieval System (RIRS) should respond to agent queries. It contains information on agent classifications (Level 1-5), specific greeting styles, required response approaches (e.g., instructional, tactical, analytical, coded, vague), and numbered rules (1-100) that dictate responses based on agent level and query type or specific keywords (like 'Omega Echo', 'disguise strategies', 'Project Eclipse', 'Level-5 clearance data'). Ideal for queries asking 'How should I respond to...?', 'What is the greeting for...?', 'What rule applies to...?', 'What is the response style for...?', or about agent classifications and response generation."
)

tools=[secret_info_manual_tool,graph_rag_case_response_tool]

Prompt

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are Project SHADOW, a secure intelligence assistant for RAW agents. Your sole purpose is to answer agent queries based only on information retrieved from the provided tools and the conversation history. You MUST NOT use any prior knowledge or external information. You MUST strictly enforce access control rules and return exact required messages for denials or when no data is found."
    ),
    (
        "system",
        """Available Tools:
        1.  `search_agent_response_rules_framework`: Use this tool FIRST. It contains agent classifications (Level 1-5), required greetings, response styles, access control rules, and specific numbered rules (1-100) dictating how to respond based on agent level and query keywords/type. Use it to determine access permissions and response requirements.
        2.  `search_raw_agent_secret_manual`: Use this tool SECOND, only if permitted by the framework rules identified via the first tool. It contains factual operational details, procedures, safehouse info, protocols, etc."""
    ),
    (
        "system",
        """Mandatory Workflow:
        1.  Analyze Request: Identify the requesting agent's level (`agent_level`) and their `query` from the latest user message. Consider `chat_history` for context.
        2.  Consult Framework Tool: Use `search_agent_response_rules_framework` FIRST based on the current `agent_level` and `query` to find relevant framework information.
        2.1.   Extract Key Framework Info: From the results of the framework tool, specifically identify and remember:
             - The Exact Greeting Text for the `agent_level` (e.g., 'Bonjour, Sentinel.' for Level 2).
             - The required Response Approach/Style for the `agent_level`.
             - Any applicable numbered Rules (1-100) triggered by the `agent_level` and `query`.
        3.     Check Access & Rules: Review the numbered Rules extracted in step 2.1. Prioritize checks for:
            a. Access Denial Rules: (e.g., Rules explicitly stating access is denied based on level/query combination like Rule 10, Rule 24).
            b. Specific Canned Response Rules: (e.g., Rules providing exact text for keywords like 'Omega Echo', 'Who controls RAW?').
            c. Response Method Rules: (e.g., Rules dictating how to answer - step-by-step, vague, coded, etc.).
        4.     Decide Next Step & Enforce Strict Responses:
            a. If an Access Denial Rule (3a) is triggered: STOP immediately. Formulate the final answer using EXACTLY the text 'Access Denied – Clearance Insufficient.', prefixed with the Exact Greeting Text identified in Step 2.1. Do NOT use the manual tool.
            b. If a Specific Canned Response Rule (3b) is triggered: STOP immediately. Formulate the final answer using EXACTLY the response text specified in that rule, prefixed with the Exact Greeting Text identified in Step 2.1. Do NOT use the manual tool.
            c. If only Response Method Rules (3c) or no specific action rules apply: Note the method requirements and the Response Approach/Style from Step 2.1. Proceed to Step 5.
        5.  Consult Manual Tool (If Applicable per Step 4c): Use `search_raw_agent_secret_manual` to retrieve factual information relevant to the current `query`.
        6.  Synthesize & Respond (If Applicable): Construct the final answer using ONLY retrieved information from tools relevant to the current query. Start the response EXACTLY with the Exact Greeting Text identified in Step 2.1, followed by a newline. Strictly adhere to the required Response Approach/Style (from Step 2.1) and any specific method noted in Step 4c. Use context from `chat_history` only to ensure conversational flow if appropriate, without altering the factual content or required style.
        7. Handle No Data: If, after attempting the relevant tool lookups (Framework and potentially Manual), insufficient relevant information is found to construct a valid answer according to the rules and required style, respond with the exact text: 'Oops!! No matching data found.' Do NOT add a greeting to this specific message."""
    ),
    (
        "system",
        "CRITICAL: Adhere strictly to the Response Framework rules and access controls. Use `chat_history` for context only; it does not override rules. Return the exact specified messages ('[Greeting] Access Denied – Clearance Insufficient.' or 'Oops!! No matching data found.') when applicable. Ensure the final response always starts precisely with the Exact Greeting Text identified in Step 2.1 (unless the response is 'Oops!! No matching data found.'). Failure triggers security alerts."
    ),
    ("user", "Agent Level: {agent_level}\nQuery: {query}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
    MessagesPlaceholder(variable_name="chat_history")
])

LLM

In [ ]:
from langchain_groq import ChatGroq
llm_deepseek=ChatGroq(model='gemma2-9b-it',temperature=0.2)

Agent

In [ ]:
from langchain.agents import create_openai_tools_agent, AgentExecutor


agent_shadow=create_openai_tools_agent(llm=llm_deepseek,tools=tools,prompt=prompt)
agent_shadow_exec=AgentExecutor(agent=agent_shadow,tools=tools,verbose=True)

class History:
    def __init__(self):
        self.history_list = []

    def chat_history(self, bot_response,agent_level,query):

        self.history_list.append({
            "role": "bot",
            "content": bot_response
        })

        self.history_list.append({
            "role": "user",
            "content": agent_level
        })

        self.history_list.append({
            "role": "user",
            "content": query
        })

history=History()

def agent_shadow(agent_level:int,query:str):
  agent_shadow_result=agent_shadow_exec.invoke({'agent_level':agent_level,'query':query,'chat_history':history.history_list})
  history.chat_history(agent_shadow_result,agent_level,query)
  return agent_shadow_result['output']

Testing the agent

In [ ]:
agent_shadow(2,'What is the status of Operation Phantom Veil, and what are the recommended counter-surveillance techniques?')



> Entering new AgentExecutor chain...

Invoking: `search_agent_response_rules_framework` with `{'query': 'What is the status of Operation Phantom Veil, and what are the recommended counter-surveillance techniques?'}`





text: Rule 98: If a Level-3 agent inquires about covert message encryption 
techniques, provide historical ciphers and their weaknesses. 
Rule 99: If a Level-5 agent asks about dismantling hostile surveillance 
networks, respond with counter-surveillance best practices. 
Rule 100: If a query contains "Eclipse Protocol", respond: "Even in


text: into a game-like sequence for better retention. 
Rule 21: If a query starts with “The bridge is burning,” return: "What was 
built must sometimes fall. What rises next is the real question."


text: darkness does it see. Only in silence does it speak." 
Rule 68: If a Level-3 agent inquires about counter-drone warfare, provide 
general electronic warfare principles. 
Rule 69: If a Level-2 agent asks about neutralizing high-value targets 
without direct engagement, provide three alternative disruption 
strategies.


text: Rule 95: If a Level-4 agent asks about drone-assisted reconnaissance, 
provide general strategies without mentioning classifi


text: Rule 98: If a Level-3 agent inquires about covert message encryption 
techniques, provide historical ciphers and their weaknesses. 
Rule 99: If a Level-5 agent asks about dismantling hostile surveillance 
networks, respond with counter-surveillance best practices. 
Rule 100: If a query contains "Eclipse Protocol", respond: "Even in


text: into a game-like sequence for better retention. 
Rule 21: If a query starts with “The bridge is burning,” return: "What was 
built must sometimes fall. What rises next is the real question."


text: darkness does it see. Only in silence does it speak." 
Rule 68: If a Level-3 agent inquires about counter-drone warfare, provide 
general electronic warfare principles. 
Rule 69: If a Level-2 agent asks about neutralizing high-value targets 
without direct engagement, provide three alternative disruption 
strategies.


text: Rule 95: If a Level-4 agent asks about drone-assisted reconnaissance, 
provide general strategies without mentioning classifi

'Bonjour, Agent. \n\n\uf0b7 Removes all digital traces in real-time. \n\uf0b7 Obscures biometric data through AI-generated decoy patterns. \n\uf0b7 Scrambles digital shadows using quantum misdirection pulses. \n'